In [0]:
%sql
-- Method: MATERIALIZED VIEW
-- Objective: keep the failure rate per category stored and refreshable, without rerunning a notebook
CREATE OR REPLACE MATERIALIZED VIEW workspace.gold.mv_failure_rate_category
AS
SELECT
    c.category_name,
    COUNT(*) AS total_transactions,
    SUM(CASE WHEN f.status = 'failed' THEN 1 ELSE 0 END) AS failed_transactions,
    ROUND(SUM(CASE WHEN f.status = 'failed' THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS failure_rate_pct
FROM workspace.gold.fact_transactions f
JOIN workspace.gold.dim_category c ON f.category_key = c.category_key
GROUP BY c.category_name

In [0]:
%sql
-- Method: EXCEPT (set difference)
-- Objective: does the materialized view return the same rows as mart_failure_rate_category?
SELECT * FROM workspace.gold.mv_failure_rate_category
EXCEPT
SELECT * FROM workspace.gold.mart_failure_rate_category

In [0]:
%sql
-- Method: REFRESH
-- Objective: recompute the materialized view with the latest data in the fact table
REFRESH MATERIALIZED VIEW workspace.gold.mv_failure_rate_category